In [1]:
import pandas as pd
import numpy as np
from tabpfn import TabPFNRegressor
from sklearn.metrics import r2_score
from scipy import stats
import os
import torch
import random
from itertools import combinations
from scipy.stats import spearmanr, wilcoxon
from scipy import sparse
import re

/disk1/anaconda3/envs/kcat/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Preparation

In [2]:
df = pd.read_csv('data/alpha_amylase_combined.csv')
proteins = np.load('embeddings/alpha_amylase_prott5_features.npy')

In [3]:
df.groupby('num_mutations').describe()

dataset                                              expression  \
                count      mean       std  min  25%  50%  75%  max      count   
num_mutations                                                                   
1              7576.0  1.002376  0.048689  1.0  1.0  1.0  1.0  2.0     7576.0   
2               103.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      103.0   
3               161.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      161.0   
4               191.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      191.0   
5               176.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      176.0   
6               274.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      274.0   
7               146.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0      146.0   
8                83.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0       83.0   
9                36.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0       36.0   
10               18.0  2.000000  0.000000  2.0  2.0  2.0  2.0  2.0       18.0   
11                1.0  2.000000       NaN  2.0  2.0  2.0  2.0  2.0        1.0   

                         ... thermostability         specific activity  \
                   mean  ...             75%     max             count   
num_mutations            ...                                             
1              0.621117  ...         0.95000  49.180            7576.0   
2              0.670155  ...         2.98650   5.316             103.0   
3              0.682528  ...         2.98925   5.752             161.0   
4              0.675424  ...         2.95650  40.374             191.0   
5              0.718477  ...         4.23550  43.258             176.0   
6              0.550358  ...         3.88525  46.847             274.0   
7              0.620705  ...         8.12600  48.911             146.0   
8              0.588988  ...         7.71100  45.077              83.0   
9              0.502778  ...        13.68475  40.026              36.0   
10             0.451778  ...         7.36100  39.363              18.0   
11             0.167000  ...         5.16500   5.165               1.0   

                                                                           
                   mean       std    min     25%     50%      75%     max  
num_mutations                                                              
1              0.767186  0.619451  0.050  0.1875  0.9100  1.04000  19.260  
2              0.914777  0.687433  0.002  0.2485  0.8820  1.46750   2.557  
3              0.954224  0.754759  0.004  0.1980  0.9670  1.57400   2.518  
4              0.948403  0.824778  0.001  0.2245  0.8580  1.63150   4.667  
5              1.226153  1.150806  0.001  0.3765  0.9915  1.71650   8.289  
6              1.193109  1.052428  0.001  0.4685  0.9930  1.52075   6.494  
7              1.896397  1.698416  0.035  0.8930  1.3030  2.02950  10.553  
8              1.930747  1.876468  0.004  0.8940  1.2150  1.92250   9.338  
9              2.083472  2.225594  0.390  0.8960  1.1485  1.98250  11.216  
10             2.021000  1.598659  0.758  0.9545  1.2270  2.53500   5.610  
11             1.690000       NaN  1.690  1.6900  1.6900  1.69000   1.690  

[11 rows x 32 columns]

In [4]:
def set_seed(seed):
    np.seterr(all="ignore")
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if use multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
# configs
SEEDS = np.random.choice(10000, size=5, replace=False).tolist()
TABPFN_DEVICE = "cuda"
MUT_COL = 'mutant'
TARGET_COL = 'specific activity'
ORDER_COL = "num_mutations"
SEP = "-"

In [6]:
# metrics
def _r2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ss = ((a - a.mean()) ** 2).sum()
    return 1 - ((a - b) ** 2).sum() / ss if ss > 0 else np.nan

def _sp(a, b):
    return spearmanr(a, b).correlation

def _nrmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sd = a.std()
    return np.sqrt(((a - b) ** 2).mean()) / sd if sd > 0 else np.nan

## Exp 1. Common target
For each target mutation order t ∈ {2, 3, 4, 5}, the test set is fixed to all variants of exactly order t. Two training conditions are evaluated: <br>

(a) lower-order — 300 randomly sampled variants with order < t <br>
(b) higher-order — 300 randomly sampled variants with order > t <br>

Both models are trained independently and evaluated on the identical test set, making R² and Spearman ρ directly comparable between conditions. This design isolates the effect of training data mutation order on predictive generalization to order-t variants. All metrics are averaged over 5 random seeds (mean ± std).

In [7]:
TRAIN_SIZE = 300

def _mutation_overlap(test_set, train_set):
    if not test_set:
        return np.nan
    return len(test_set & train_set) / len(test_set)

def _position_overlap(test_muts, train_muts):
    def to_pos_set(muts):
        pos = set()
        for m in muts:
            try:
                pos.add(int(m[1:-1]))
            except ValueError:
                pass
        return pos

    test_pos  = to_pos_set(test_muts)
    train_pos = to_pos_set(train_muts)
    if not test_pos:
        return np.nan
    return len(test_pos & train_pos) / len(test_pos)


def analyze_order_k(df, k, features=None, y=None):
    """
    Parameters
    ----------
    df       : DataFrame with columns ['mutant', 'num_mutations']
    k        : target mutation order
    features : array-like, shape (len(df), ...) — optional
    y        : array-like, shape (len(df),)      — optional

    Returns
    -------
    raw_df   : DataFrame with per-seed raw results
    summary  : DataFrame with mean ± std
    """
    test_idx  = df.index[df.num_mutations == k].tolist()
    low_pool  = df.index[df.num_mutations <  k].tolist()
    high_pool = df.index[df.num_mutations >  k].tolist()

    test_muts = set()
    for m in df.loc[test_idx, 'mutant']:
        test_muts.update(m.split('-'))

    records = []

    for seed in SEEDS:
        rng = np.random.default_rng(seed)

        n_low  = min(TRAIN_SIZE, len(low_pool))
        n_high = min(TRAIN_SIZE, len(high_pool))

        low_idx  = rng.choice(low_pool,  size=n_low,  replace=False).tolist()
        high_idx = rng.choice(high_pool, size=n_high, replace=False).tolist()

        low_muts  = set()
        high_muts = set()
        for m in df.loc[low_idx,  'mutant']: low_muts.update(m.split('-'))
        for m in df.loc[high_idx, 'mutant']: high_muts.update(m.split('-'))

        low_mut_cov  = _mutation_overlap(test_muts, low_muts)
        high_mut_cov = _mutation_overlap(test_muts, high_muts)
        low_pos_cov  = _position_overlap(test_muts, low_muts)
        high_pos_cov = _position_overlap(test_muts, high_muts)

        rec = {
            'seed': seed,
            'low_mutation_coverage':  low_mut_cov,
            'high_mutation_coverage': high_mut_cov,
            'low_position_coverage':  low_pos_cov,
            'high_position_coverage': high_pos_cov,
        }

        if features is not None and y is not None:
            X  = np.array(features)
            yy = np.array(y)

            X_test,  y_test  = X[test_idx],  yy[test_idx]
            X_low,   y_low   = X[low_idx],   yy[low_idx]
            X_high,  y_high  = X[high_idx],  yy[high_idx]

            for tag, X_tr, y_tr in [('low', X_low, y_low), ('high', X_high, y_high)]:
                mask_tr   = ~np.isnan(y_tr)
                mask_test = ~np.isnan(y_test)

                model = TabPFNRegressor(device='cuda', ignore_pretraining_limits=True, random_state=seed)
                model.fit(X_tr[mask_tr], y_tr[mask_tr])
                preds = model.predict(X_test[mask_test])

                rec[f'{tag}_r2']       = r2_score(y_test[mask_test], preds)
                rec[f'{tag}_spearman'] = spearmanr(preds, y_test[mask_test]).correlation

        records.append(rec)

    raw_df = pd.DataFrame(records).set_index('seed')

    summary = pd.concat([
        raw_df.mean().rename('mean'),
        raw_df.std().rename('std'),
    ], axis=1)

    return raw_df, summary


In [8]:
all_results = {}
for k in [2, 3, 4, 5]:
    raw, summary = analyze_order_k(df, k=k, features=proteins, y=df['specific activity'].values)
    all_results[k] = {'raw': raw, 'summary': summary}
    print(f"k={k} done")

metrics = [
    'low_r2', 'high_r2',
    'low_spearman', 'high_spearman',
    'low_mutation_coverage', 'high_mutation_coverage',
    'low_position_coverage', 'high_position_coverage',
]

rows = []
for k, res in all_results.items():
    row = {'order': k}
    for m in metrics:
        if m in res['summary'].index:
            row[f'{m}_mean'] = res['summary'].loc[m, 'mean']
            row[f'{m}_std']  = res['summary'].loc[m, 'std']
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('order')

def fmt(df, col):
    m_col, s_col = f'{col}_mean', f'{col}_std'
    if m_col not in df.columns:
        return pd.Series([float('nan')] * len(df), index=df.index)
    return df.apply(lambda r: f"{r[m_col]:.3f} ± {r[s_col]:.3f}", axis=1)

display_df = pd.DataFrame({
    'low_R²':        fmt(summary_df, 'low_r2'),
    'high_R²':       fmt(summary_df, 'high_r2'),
    'low_Spearman':  fmt(summary_df, 'low_spearman'),
    'high_Spearman': fmt(summary_df, 'high_spearman'),
    'low_mut_cov':   fmt(summary_df, 'low_mutation_coverage'),
    'high_mut_cov':  fmt(summary_df, 'high_mutation_coverage'),
    'low_pos_cov':   fmt(summary_df, 'low_position_coverage'),
    'high_pos_cov':  fmt(summary_df, 'high_position_coverage'),
}, index=summary_df.index)

display_df.index.name = 'order (t)'
display_df

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


k=2 done


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


k=3 done


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


k=4 done


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


k=5 done


,low_R²,high_R²,low_Spearman,high_Spearman,low_mut_cov,high_mut_cov,low_pos_cov,high_pos_cov
order (t),,,,,,,,
2,0.033 ± 0.068,0.457 ± 0.058,0.283 ± 0.115,0.692 ± 0.040,0.056 ± 0.026,0.715 ± 0.015,0.557 ± 0.049,1.000 ± 0.000
3,-0.086 ± 0.133,0.560 ± 0.036,0.033 ± 0.175,0.742 ± 0.024,0.144 ± 0.051,0.934 ± 0.016,0.596 ± 0.082,0.996 ± 0.010
4,-0.004 ± 0.064,0.519 ± 0.047,0.259 ± 0.098,0.687 ± 0.018,0.335 ± 0.079,0.932 ± 0.018,0.690 ± 0.064,0.996 ± 0.009
5,-0.073 ± 0.100,0.530 ± 0.063,0.363 ± 0.105,0.609 ± 0.049,0.439 ± 0.091,0.922 ± 0.018,0.761 ± 0.075,0.989 ± 0.010


## Exp 2. Mutational distance and mutational order
We trained TabPFN on a split of protein variants and evaluated generalization with respect to three structural properties of the test set, using five fixed random seeds (mean ± std).

For each test variant, we computed its mutational distance to the nearest training variant — defined as the minimum Hamming distance in mutation-token space 

Three analyses were conducted, all using mutational distance as a primary control:

A1 (Distance-Decay): R² and Spearman ρ as a function of mutational distance bin (0–4, 5+).<br>
A2 (Order Effect): Within each distance bin, performance broken down by mutation order; additionally, Spearman ρ between mutation order and absolute prediction error quantifies the residual order effect after controlling for distance.

In [9]:
TRAIN_MAX_ORDER = 2     # extrapolate mode only
TEST_FRAC = 0.2
TRAIN_SIZE = 300
PAIR_FULL_THRESHOLD = 1.0
MIN_BIN_N = 10

tok_list = [frozenset(str(m).split(SEP)) for m in df[MUT_COL]]
y = df[TARGET_COL].to_numpy(float)
order = df[ORDER_COL].to_numpy(int)
N = len(df)

vocab = {}
for ts in tok_list:
    for t in ts:
        vocab.setdefault(t, len(vocab))

def _multihot(idxs):
    rows, cols = [], []
    for r, i in enumerate(idxs):
        for t in tok_list[i]:
            rows.append(r)
            cols.append(vocab[t])
    return sparse.csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(len(idxs), len(vocab))
    )

def _agg(rows, group_cols, val_cols):
    df_ = pd.DataFrame(rows)
    parts = [
        df_.groupby(group_cols)[c].agg(
            **{f'{c}_mean': 'mean',
               f'{c}_std': lambda x: x.std(ddof=0),
               f'{c}_seeds': 'count'}
        )
        for c in val_cols
    ]
    return pd.concat(parts, axis=1).round(3)

In [10]:
# random split
A1_rows, A2_rows = [], []

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    _test = []
    for o in np.unique(order):
        io = np.where(order == o)[0]
        nt = int(round(TEST_FRAC * len(io)))
        if len(io) > 1 and nt > 0:
            _test += list(rng.choice(io, nt, replace=False))
    test_idx  = np.array(sorted(_test))
    mask      = np.ones(N, bool); mask[test_idx] = False
    train_idx = np.where(mask)[0]

    model = TabPFNRegressor(device=TABPFN_DEVICE,
                            ignore_pretraining_limits=True,
                            random_state=int(seed))
    model.fit(proteins[train_idx], y[train_idx])
    pred   = model.predict(proteins[test_idx])
    yt     = y[test_idx]
    ot     = order[test_idx]
    abserr = np.abs(yt - pred)

    # mutational distance to nearest train
    Xtr, Xte = _multihot(train_idx), _multihot(test_idx)
    a_ = np.asarray(Xte.sum(1)).ravel()
    b_ = np.asarray(Xtr.sum(1)).ravel()
    inter    = (Xte @ Xtr.T).toarray()
    min_dist = (a_[:, None] + b_[None, :] - 2 * inter).min(axis=1).astype(int)
    del inter

    dbin = np.minimum(min_dist, 5)

    # mutational distance
    for d in sorted(set(dbin)):
        sel = dbin == d
        if sel.sum() < MIN_BIN_N:
            continue
        A1_rows.append({
            'seed': seed, 'dist': d, 'n': int(sel.sum()),
            'R2': _r2(yt[sel], pred[sel]),
            'Spearman': _sp(yt[sel], pred[sel]),
        })

    # mutational order effect at fixed distance
    for d in sorted(set(dbin)):
        seld = dbin == d
        if seld.sum() < MIN_BIN_N:
            continue
        for o in sorted(set(ot[seld])):
            s = seld & (ot == o)
            if s.sum() < MIN_BIN_N:
                continue
            A2_rows.append({
                'seed': seed, 'dist': d, 'order': o,
                'n': int(s.sum()),
                'R2': _r2(yt[s], pred[s]),
                'Spearman': _sp(yt[s], pred[s]),
            })
        if seld.sum() >= 30 and len(set(ot[seld])) > 1:
            A2_rows.append({
                'seed': seed, 'dist': d, 'order': 'rho',
                'n': int(seld.sum()), 'R2': np.nan,
                'Spearman': spearmanr(ot[seld], abserr[seld]).correlation,
            })

    print(f"seed={seed}  train={len(train_idx)}  test={len(test_idx)}  done")

a1 = _agg(A1_rows, ['dist'], ['n', 'R2', 'Spearman'])
a2 = _agg([r for r in A2_rows if r['order'] != 'rho'], ['dist', 'order'], ['n', 'R2', 'Spearman'])
a2_rho = _agg([r for r in A2_rows if r['order'] == 'rho'], ['dist'], ['n', 'Spearman'])

print("── A1: Distance-Decay ──"); display(a1)
print("── A2: Order Effect (per dist) ──"); display(a2)
print("── A2: Order vs |error| Spearman ──"); display(a2_rho)

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


seed=4192  train=7012  test=1753  done


analytics-python queue is full
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=5638  train=7012  test=1753  done
seed=3285  train=7012  test=1753  done
seed=7060  train=7012  test=1753  done
seed=3597  train=7012  test=1753  done
── A1: Distance-Decay ──


,n_mean,n_std,n_seeds,R2_mean,R2_std,R2_seeds,Spearman_mean,Spearman_std,Spearman_seeds
dist,,,,,,,,,
1,184.6,6.280,5,0.801,0.030,5,0.811,0.027,5
2,1559.8,7.467,5,0.310,0.034,5,0.614,0.015,5
3,12.0,0.000,1,0.334,0.000,1,0.594,0.000,1


── A2: Order Effect (per dist) ──


n_mean  n_std  n_seeds  R2_mean  R2_std  R2_seeds  \
dist order                                                        
1    1        13.333  2.357        3   -0.778   1.172         3   
     2        20.800  0.400        5    0.429   0.129         5   
     3        27.400  1.744        5    0.646   0.136         5   
     4        29.800  1.939        5    0.742   0.091         5   
     5        27.200  2.040        5    0.794   0.076         5   
     6        25.200  3.868        5    0.754   0.191         5   
     7        21.600  2.653        5    0.849   0.044         5   
     8        12.400  1.497        5    0.783   0.040         5   
2    1      1503.600  3.007        5    0.263   0.036         5   
     6        27.600  3.720        5    0.649   0.108         5   

            Spearman_mean  Spearman_std  Spearman_seeds  
dist order                                               
1    1              0.289         0.216               3  
     2              0.663         0.117               5  
     3              0.771         0.078               5  
     4              0.795         0.073               5  
     5              0.831         0.045               5  
     6              0.795         0.144               5  
     7              0.889         0.045               5  
     8              0.817         0.051               5  
2    1              0.606         0.010               5  
     6              0.732         0.100               5

── A2: Order vs |error| Spearman ──


,n_mean,n_std,n_seeds,Spearman_mean,Spearman_std,Spearman_seeds
dist,,,,,,
1,184.6,6.280,5,0.001,0.038,5
2,1559.8,7.467,5,0.125,0.022,5


In [11]:
# extrapolate split
A1_rows, A2_rows = [], []

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    test_idx  = np.where(order >  TRAIN_MAX_ORDER)[0]
    train_idx = np.where(order <= TRAIN_MAX_ORDER)[0]

    model = TabPFNRegressor(device=TABPFN_DEVICE,
                            ignore_pretraining_limits=True,
                            random_state=int(seed))
    model.fit(proteins[train_idx], y[train_idx])
    pred   = model.predict(proteins[test_idx])
    yt     = y[test_idx]
    ot     = order[test_idx]
    abserr = np.abs(yt - pred)

    # mutational distance to nearest train
    Xtr, Xte = _multihot(train_idx), _multihot(test_idx)
    a_ = np.asarray(Xte.sum(1)).ravel()
    b_ = np.asarray(Xtr.sum(1)).ravel()
    inter    = (Xte @ Xtr.T).toarray()
    min_dist = (a_[:, None] + b_[None, :] - 2 * inter).min(axis=1).astype(int)
    del inter

    dbin = np.minimum(min_dist, 5)

    # mutational distance
    for d in sorted(set(dbin)):
        sel = dbin == d
        if sel.sum() < MIN_BIN_N:
            continue
        A1_rows.append({
            'seed': seed, 'dist': d, 'n': int(sel.sum()),
            'R2': _r2(yt[sel], pred[sel]),
            'Spearman': _sp(yt[sel], pred[sel]),
        })

    # mutational order effect at fixed distance
    for d in sorted(set(dbin)):
        seld = dbin == d
        if seld.sum() < MIN_BIN_N:
            continue
        for o in sorted(set(ot[seld])):
            s = seld & (ot == o)
            if s.sum() < MIN_BIN_N:
                continue
            A2_rows.append({
                'seed': seed, 'dist': d, 'order': o,
                'n': int(s.sum()),
                'R2': _r2(yt[s], pred[s]),
                'Spearman': _sp(yt[s], pred[s]),
            })
        if seld.sum() >= 30 and len(set(ot[seld])) > 1:
            A2_rows.append({
                'seed': seed, 'dist': d, 'order': 'rho',
                'n': int(seld.sum()), 'R2': np.nan,
                'Spearman': spearmanr(ot[seld], abserr[seld]).correlation,
            })

    print(f"seed={seed}  train={len(train_idx)}  test={len(test_idx)}  done")

a1 = _agg(A1_rows, ['dist'], ['n', 'R2', 'Spearman'])
a2 = _agg([r for r in A2_rows if r['order'] != 'rho'], ['dist', 'order'], ['n', 'R2', 'Spearman'])
a2_rho = _agg([r for r in A2_rows if r['order'] == 'rho'], ['dist'], ['n', 'Spearman'])

print("── A1: Distance-Decay ──"); display(a1)
print("── A2: Order Effect (per dist) ──"); display(a2)
print("── A2: Order vs |error| Spearman ──"); display(a2_rho)

analytics-python queue is full
analytics-python queue is full
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=4192  train=7679  test=1086  done
seed=5638  train=7679  test=1086  done
seed=3285  train=7679  test=1086  done


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=7060  train=7679  test=1086  done
seed=3597  train=7679  test=1086  done
── A1: Distance-Decay ──


,n_mean,n_std,n_seeds,R2_mean,R2_std,R2_seeds,Spearman_mean,Spearman_std,Spearman_seeds
dist,,,,,,,,,
1,125.0,0.0,5,0.400,0.009,5,0.713,0.007,5
2,192.0,0.0,5,0.265,0.009,5,0.577,0.014,5
3,162.0,0.0,5,0.087,0.012,5,0.416,0.022,5
4,123.0,0.0,5,-0.026,0.018,5,0.379,0.036,5
5,484.0,0.0,5,-0.245,0.030,5,-0.056,0.092,5


── A2: Order Effect (per dist) ──


n_mean  n_std  n_seeds  R2_mean  R2_std  R2_seeds  Spearman_mean  \
dist order                                                                     
1    3       125.0    0.0        5    0.400   0.009         5          0.713   
2    3        36.0    0.0        5   -0.005   0.021         5          0.080   
     4       156.0    0.0        5    0.313   0.008         5          0.651   
3    4        35.0    0.0        5   -0.200   0.021         5          0.310   
     5       127.0    0.0        5    0.138   0.011         5          0.422   
4    5        49.0    0.0        5   -0.247   0.027         5          0.345   
     6        74.0    0.0        5    0.119   0.010         5          0.421   
5    6       200.0    0.0        5   -0.095   0.030         5          0.037   
     7       146.0    0.0        5   -0.362   0.034         5          0.045   
     8        83.0    0.0        5   -0.342   0.032         5         -0.074   
     9        36.0    0.0        5   -0.321   0.028         5         -0.126   
     10       18.0    0.0        5   -0.608   0.050         5         -0.664   

            Spearman_std  Spearman_seeds  
dist order                                
1    3             0.007               5  
2    3             0.071               5  
     4             0.008               5  
3    4             0.084               5  
     5             0.015               5  
4    5             0.060               5  
     6             0.017               5  
5    6             0.097               5  
     7             0.092               5  
     8             0.085               5  
     9             0.093               5  
     10            0.075               5

── A2: Order vs |error| Spearman ──


,n_mean,n_std,n_seeds,Spearman_mean,Spearman_std,Spearman_seeds
dist,,,,,,
2,192.0,0.0,5,-0.130,0.010,5
3,162.0,0.0,5,-0.116,0.011,5
4,123.0,0.0,5,0.028,0.029,5
5,484.0,0.0,5,-0.044,0.014,5


## Exp 3. Single to multi-mutation variants
We trained TabPFN on all order-1 (single-mutation) variants and evaluated generalization to higher-order variants (k = 2–8). For each order k, we stratified the test variants into two groups based on single-mutation coverage: 

(a) fully covered — all constituent single mutations of the variant appear in the training set<br>
(b) partially covered — at least one constituent single mutation is absent from the training set

R² and Spearman ρ were computed separately for each group to assess whether complete coverage of individual mutations is sufficient for accurate prediction of higher-order combinations.

In [12]:
MIN_N = 10 

measured = set(df.loc[df.num_mutations == 1, 'mutant'])

# train model
single_index = df[df.num_mutations == 1].index.tolist()
single_X, single_y = proteins[single_index], df['specific activity'][single_index].tolist()
single_model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=79238)
single_model.fit(single_X, single_y)

def _is_fully_covered(mutant_str):
    return all(m in measured for m in mutant_str.split('-'))

strata = {}
for k in range(2, 9):
    sub = df[df.num_mutations == k]
    mask = sub['mutant'].apply(_is_fully_covered)
    strata[k] = {'fully': sub[mask], 'partial': sub[~mask]}

def _eval(idx):
    if len(idx) < MIN_N:
        return None, None
    y    = np.array(df.loc[idx, 'specific activity'], dtype=float)
    pred = single_model.predict(proteins[idx])
    mask = ~np.isnan(y) & ~np.isnan(pred)
    if mask.sum() < 2:
        return None, None
    return (r2_score(y[mask], pred[mask]),
            stats.spearmanr(pred[mask], y[mask]).correlation)

rows = []
for k, st in strata.items():
    r2_full, sp_full   = _eval(st['fully'].index.tolist())
    r2_part, sp_part   = _eval(st['partial'].index.tolist())
    rows.append({
        'order (k)':        k,
        'n_fully':          len(st['fully']),
        'n_partial':        len(st['partial']),
        'R²_fully':         r2_full,
        'R²_partial':       r2_part,
        'Spearman_fully':   sp_full,
        'Spearman_partial': sp_part,
    })

exp3 = pd.DataFrame(rows).set_index('order (k)')
exp3

error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


,n_fully,n_partial,R²_fully,R²_partial,Spearman_fully,Spearman_partial
order (k),,,,,,
2,102,1,0.079438,NaN,0.380247,NaN
3,161,0,-0.028960,NaN,0.077573,NaN
4,184,7,-0.001413,NaN,0.154074,NaN
5,143,33,-0.138555,-0.384607,-0.125846,0.328877
6,200,74,-0.083525,-0.342071,-0.055746,0.187681
7,45,101,-0.644553,-0.410781,-0.395599,0.063489
8,16,67,-0.838889,-0.370575,-0.129412,-0.198300


## Exp 4. Directional asymmetry of mutation order generalization
We assessed the directional asymmetry of mutation order generalization by defining a threshold k ∈ {1, …, 8} that partitions variants into low-order (≤ k) and high-order (> k) groups. For each k, two conditions were compared under a matched design — training and test sizes were equalized across directions to ensure a fair comparison:

Forward (low → high): trained on low-order variants, evaluated on high-order variants<br>
Reverse (high → low): trained on high-order variants, evaluated on low-order variants

TabPFN was used as the predictor. Performance was measured by R², normalized RMSE, and Spearman ρ, averaged over 5 random seeds (mean ± std).

In [13]:
KS          = [1, 2, 3, 4, 5, 6, 7, 8]
N_TEST_CAP  = 150
MIN_TEST_N  = 10
MIN_TRAIN_N = 20

y     = df[TARGET_COL].to_numpy(float)
order = df['num_mutations'].to_numpy()

def _predict(tr_idx, te_idx, seed):
    model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True,
                            random_state=seed)
    model.fit(proteins[tr_idx], y[tr_idx])
    return np.asarray(model.predict(proteins[te_idx]))

rows = []
for k in KS:
    lo_all = np.where(order <= k)[0]
    hi_all = np.where(order >  k)[0]

    for seed in SEEDS:
        rng = np.random.default_rng(1000 * k + seed)

        n_test = min(N_TEST_CAP, len(lo_all) // 2, len(hi_all) // 2)
        if n_test < MIN_TEST_N:
            continue

        test_lo = rng.choice(lo_all, n_test, replace=False)
        test_hi = rng.choice(hi_all, n_test, replace=False)

        pool_fwd = np.setdiff1d(lo_all, test_lo)  # low pool
        pool_rev = np.setdiff1d(hi_all, test_hi)  # high pool

        n_train = min(len(pool_fwd), len(pool_rev))
        if n_train < MIN_TRAIN_N:
            continue

        tr_fwd = rng.choice(pool_fwd, n_train, replace=False)
        tr_rev = rng.choice(pool_rev, n_train, replace=False)

        # forward: low → high
        pred_fwd = _predict(tr_fwd, test_hi, seed)
        yt_hi    = y[test_hi]

        # reverse: high → low
        pred_rev = _predict(tr_rev, test_lo, seed)
        yt_lo    = y[test_lo]

        rows.append({
            'k': k, 'seed': seed, 'n_train': n_train, 'n_test': n_test,
            'fwd_R2':    _r2(yt_hi, pred_fwd),
            'fwd_nRMSE': _nrmse(yt_hi, pred_fwd),
            'fwd_Sp':    _sp(yt_hi, pred_fwd),
            'rev_R2':    _r2(yt_lo, pred_rev),
            'rev_nRMSE': _nrmse(yt_lo, pred_rev),
            'rev_Sp':    _sp(yt_lo, pred_rev),
        })

raw_df = pd.DataFrame(rows)

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is

In [14]:
# summary
metrics = ['fwd_R2', 'rev_R2', 'fwd_nRMSE', 'rev_nRMSE', 'fwd_Sp', 'rev_Sp']

def fmt(g, col):
    return f"{g[col].mean():+.3f} ± {g[col].std():.3f}"

summary_rows = []
for k, g in raw_df.groupby('k'):
    summary_rows.append({
        'k':         k,
        'n_train':   int(g['n_train'].mean()),
        'n_test':    int(g['n_test'].mean()),
        'fwd_R²':    fmt(g, 'fwd_R2'),
        'rev_R²':    fmt(g, 'rev_R2'),
        'fwd_nRMSE': fmt(g, 'fwd_nRMSE'),
        'rev_nRMSE': fmt(g, 'rev_nRMSE'),
        'fwd_Sp':    fmt(g, 'fwd_Sp'),
        'rev_Sp':    fmt(g, 'rev_Sp'),
    })

summary_df = pd.DataFrame(summary_rows).set_index('k')
summary_df

,n_train,n_test,fwd_R²,rev_R²,fwd_nRMSE,rev_nRMSE,fwd_Sp,rev_Sp
k,,,,,,,,
1,1039,150,-0.179 ± 0.058,-0.258 ± 0.126,+1.085 ± 0.027,+1.120 ± 0.056,-0.078 ± 0.140,+0.154 ± 0.047
2,936,150,-0.112 ± 0.040,-0.377 ± 0.109,+1.055 ± 0.019,+1.173 ± 0.046,+0.065 ± 0.079,+0.117 ± 0.060
3,775,150,-0.128 ± 0.061,-0.185 ± 0.117,+1.062 ± 0.029,+1.088 ± 0.054,+0.289 ± 0.136,+0.105 ± 0.090
4,584,150,-0.088 ± 0.056,-0.314 ± 0.054,+1.043 ± 0.027,+1.146 ± 0.024,+0.272 ± 0.156,+0.061 ± 0.034
5,408,150,+0.034 ± 0.211,-0.549 ± 0.488,+0.978 ± 0.110,+1.233 ± 0.190,+0.389 ± 0.173,+0.071 ± 0.105
6,142,142,-0.115 ± 0.274,-0.716 ± 0.533,+1.049 ± 0.136,+1.298 ± 0.199,+0.399 ± 0.174,+0.167 ± 0.070
7,69,69,-0.194 ± 0.194,-2.219 ± 2.808,+1.090 ± 0.088,+1.670 ± 0.733,+0.293 ± 0.346,+0.070 ± 0.037
8,28,27,-0.394 ± 0.523,-85.319 ± 48.450,+1.164 ± 0.222,+9.001 ± 2.574,+0.215 ± 0.325,+0.105 ± 0.175


## Exp 5. Epistasis
We evaluated whether a purely additive model of mutational effects can explain higher-order variant activity as well as a machine learning model. A TabPFN model was trained exclusively on order-1 (single-mutation) variants. The additive baseline predicts each higher-order variant as μ + Σeffect[token], where μ is the mean activity of single mutants and effect[token] is the mean deviation of each single mutation from μ. Both models were evaluated on the subset of higher-order variants whose constituent single mutations were all observed in the training set (fully-covered variants).

Two analyses were conducted: (i) R² and Spearman ρ of ML vs. additive predictions, with bootstrap resampling (n = 1,000) to test whether the difference is significant; (ii) Spearman correlations between |ML error| and |epistasis| (= |y − additive prediction|), mutation order and |ML error|, and mutation order and |epistasis|, to assess whether prediction difficulty is driven by epistatic interactions.

In [15]:
N_BOOTS = 1000

def run_additivity_experiment(df, features, seed=42):
    set_seed(seed)
    rng = np.random.default_rng(seed)

    singles = df[df.num_mutations == 1].copy()
    mu = singles['specific activity'].mean()
    effect = (
        singles.groupby('mutant')['specific activity']
        .mean()
        .subtract(mu)
        .to_dict()
    )

    measured = set(effect.keys())
    fully = df[
        (df.num_mutations > 1) &
        df['mutant'].apply(lambda m: all(t in measured for t in m.split('-')))
    ].copy()

    fully['additive_pred'] = fully['mutant'].apply(
        lambda m: mu + sum(effect[t] for t in m.split('-'))
    )
    fully['epistasis'] = fully['specific activity'] - fully['additive_pred']

    model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True,
                            random_state=seed)
    model.fit(features[singles.index], singles['specific activity'].values)
    fully['ml_pred'] = model.predict(features[fully.index])

    fully['ml_error']       = (fully['specific activity'] - fully['ml_pred']).abs()
    fully['additive_error'] = fully['epistasis'].abs()

    y_te = fully['specific activity'].values
    y_ml = fully['ml_pred'].values
    y_ad = fully['additive_pred'].values

    boot_diff_r2, boot_diff_sp = [], []
    n = len(fully)
    for _ in range(N_BOOTS):
        idx = rng.integers(0, n, size=n)
        yt, ym, ya = y_te[idx], y_ml[idx], y_ad[idx]
        boot_diff_r2.append(r2_score(yt, ym) - r2_score(yt, ya))
        boot_diff_sp.append(
            stats.spearmanr(yt, ym).correlation -
            stats.spearmanr(yt, ya).correlation
        )

    summary_i = pd.DataFrame({
        'R²':            [r2_score(y_te, y_ml),
                          r2_score(y_te, y_ad)],
        'Spearman':      [stats.spearmanr(y_te, y_ml).correlation,
                          stats.spearmanr(y_te, y_ad).correlation],
        'P(≤ other)':    [
            (np.array(boot_diff_r2) <= 0).mean(),   # P(ML ≤ Additive)
            (np.array(boot_diff_r2) >= 0).mean(),   # P(Additive ≤ ML)
        ],
    }, index=['ML (TabPFN)', 'Additive']).round(4)

    pairs = [
        ('|epistasis| ~ |ML error|', 'additive_error', 'ml_error'),
        ('order ~ |ML error|',       'num_mutations',  'ml_error'),
        ('order ~ |epistasis|',      'num_mutations',  'additive_error'),
    ]
    summary_ii = pd.DataFrame(
        [{'pair': label,
          'Spearman r': stats.spearmanr(fully[a], fully[b]).correlation,
          'p-value':    stats.spearmanr(fully[a], fully[b]).pvalue}
         for label, a, b in pairs]
    ).set_index('pair').round(4)

    return fully, summary_i, summary_ii

In [16]:
fully, summary_i, summary_ii = run_additivity_experiment(df, proteins, seed=1024)

print(f"Test set: {len(fully)} fully-covered multi-mutants "
      f"(order {fully.num_mutations.min()}–{fully.num_mutations.max()})\n")

print("── (i) ML vs Additive ──────────────────────────────────────")
display(summary_i)

print("\n── (ii) Error correlations ─────────────────────────────────")
display(summary_ii)

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


Test set: 853 fully-covered multi-mutants (order 2–10)

── (i) ML vs Additive ──────────────────────────────────────


,R²,Spearman,P(≤ other)
ML (TabPFN),-0.0767,-0.0291,0.0
Additive,-4.0115,0.5124,1.0



── (ii) Error correlations ─────────────────────────────────


,Spearman r,p-value
pair,,
|epistasis| ~ |ML error|,-0.1191,0.0005
order ~ |ML error|,-0.0183,0.5935
order ~ |epistasis|,0.5857,0.0000


## Exp 6. Pair-covered vs. Pair-uncovered
We tested whether including pair-covered carrier variants in the training set improves prediction of order-2 test variants. The test set was fixed to order-2 variants whose constituent single mutations were both observed, and for which at least one other higher-order variant sharing the same mutation pair (a carrier) exists in the dataset. Two training conditions were compared:

* Train A (pair-covered): a base set of single-mutation variants (up to n = 2,000) augmented with one randomly selected carrier per test pair<br>
* Train B (pair-uncovered): the same base singles augmented with matched-size higher-order filler variants that share no mutation pairs with the test set

Train sizes were equalized between conditions. A TabPFN model was fit under each condition and evaluated on the identical test set using R², Spearman ρ, and nRMSE. Statistical significance of the A vs. B difference was assessed with a Wilcoxon signed-rank test across 5 random seeds.

In [17]:
BASE_SINGLE = 2000
TEST_N      = 150

MUT_RE = re.compile(r'^([A-Za-z])(\d+)([A-Za-z])$')

def _parse(s):
    out = set()
    for t in str(s).split('-'):
        m = MUT_RE.match(t.strip())
        if m:
            out.add((int(m.group(2)), m.group(3).upper()))
    return frozenset(out)

subs   = [_parse(s) for s in df['mutant']]
orders = df['num_mutations'].to_numpy()
y      = df[TARGET_COL].to_numpy(float)

single_row = {}
for i, s in enumerate(subs):
    if orders[i] == 1:
        single_row.setdefault(next(iter(s)), i)

pair_to_rows = {}
for i, s in enumerate(subs):
    if orders[i] >= 2:
        for a, b in combinations(sorted(s), 2):
            pair_to_rows.setdefault((a, b), []).append(i)

elig = []
for i in np.where(orders == 2)[0]:
    pr = sorted(subs[i])
    if len(pr) == 2 and pr[0] in single_row and pr[1] in single_row:
        carr = [r for r in pair_to_rows.get((pr[0], pr[1]), []) if r != i]
        if carr:
            elig.append((i, (pr[0], pr[1]), carr))

print(f"Eligible order-2 test variants: {len(elig)}")

Eligible order-2 test variants: 79


In [18]:
def _predict(X_tr, y_tr, X_te, seed):
    model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True,
                            random_state=seed)
    model.fit(X_tr, y_tr)
    return np.asarray(model.predict(X_te))

single_all = np.where(orders == 1)[0]
master     = np.random.default_rng(0)
test_n     = min(TEST_N, len(elig))
rows       = []

for seed in SEEDS:
    rng    = np.random.default_rng(master.integers(1 << 30))
    chosen = [elig[j] for j in rng.choice(len(elig), test_n, replace=False)]

    test_idx = np.array([c[0] for c in chosen])
    ts       = set(test_idx.tolist())
    tpairs   = {(a, b) for _, (a, b), _ in chosen}

    const  = {single_row[a] for _, (a, b), _ in chosen} | \
             {single_row[b] for _, (a, b), _ in chosen}
    other  = [r for r in single_all if r not in const]
    keep   = min(max(0, BASE_SINGLE - len(const)), len(other))
    extra  = list(rng.choice(other, keep, replace=False)) if keep > 0 else []
    base   = [r for r in list(const) + extra if r not in ts]

    carriers = []
    for _, (a, b), cands in chosen:
        c = [r for r in cands if r not in ts]
        if c:
            carriers.append(int(rng.choice(c)))
    carriers = list(dict.fromkeys(carriers))
    train_A  = list(dict.fromkeys(base + carriers))

    fill = [
        r for r in np.where(orders >= 2)[0]
        if r not in ts and not any(
            (x, z) in tpairs
            for x, z in combinations(sorted(subs[r]), 2)
        )
    ]
    fillers = list(rng.choice(fill, min(len(carriers), len(fill)), replace=False)) \
              if fill else []
    train_B = list(dict.fromkeys(base + fillers))

    m = min(len(train_A), len(train_B))
    train_A, train_B = train_A[:m], train_B[:m]

    yt   = y[test_idx]
    y_A  = _predict(proteins[train_A], y[train_A], proteins[test_idx], seed)
    y_B  = _predict(proteins[train_B], y[train_B], proteins[test_idx], seed)

    rows.append({
        'n_test':    len(test_idx),
        'n_train':   m,
        'n_carrier': len(carriers),
        'n_filler':  len(fillers),
        'A_R²':      _r2(yt, y_A),
        'B_R²':      _r2(yt, y_B),
        'A_Sp':      _sp(yt, y_A),
        'B_Sp':      _sp(yt, y_B),
        'A_nRMSE':   _nrmse(yt, y_A),
        'B_nRMSE':   _nrmse(yt, y_B),
    })

raw_df = pd.DataFrame(rows)

analytics-python queue is full
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


In [19]:
metrics = [
    ('R²',       'A_R²',    'B_R²',    'greater'),  
    ('Spearman', 'A_Sp',    'B_Sp',    'greater'),
    ('nRMSE',    'A_nRMSE', 'B_nRMSE', 'less'),     
]

wilcoxon_rows = []
for label, col_a, col_b, alt in metrics:
    try:
        p = wilcoxon(raw_df[col_a], raw_df[col_b],
                     alternative=alt,
                     zero_method='wilcox',
                     method='exact').pvalue
    except ValueError:
        p = np.nan
    wilcoxon_rows.append({
        'metric':     label,
        'A mean±std': f"{raw_df[col_a].mean():+.3f} ± {raw_df[col_a].std():.3f}",
        'B mean±std': f"{raw_df[col_b].mean():+.3f} ± {raw_df[col_b].std():.3f}",
        'mean(A−B)':  round((raw_df[col_a] - raw_df[col_b]).mean(), 4),
        'p (one-sided)': round(p, 4),
    })

summary_df = pd.DataFrame(wilcoxon_rows).set_index('metric')
display(summary_df)

,A mean±std,B mean±std,mean(A−B),p (one-sided)
metric,,,,
R²,+0.335 ± 0.078,+0.150 ± 0.044,0.1847,0.0312
Spearman,+0.679 ± 0.047,+0.458 ± 0.047,0.2213,0.0312
nRMSE,+0.814 ± 0.047,+0.922 ± 0.024,-0.1072,0.0312


## Exp 7. Exact substitution vs. Position on single-mutation variants
We examined how prediction accuracy on single-mutation variants degrades as the degree of training-set coverage decreases. For each seed, 15% of all mutation positions were randomly held out, and all variants involving those positions were excluded from the training set. The test set comprised all single mutants at held-out positions plus 30% of single mutants at non-held positions. Test variants were stratified into three classes based on their relationship to the training set:

* exact_seen — the identical substitution (position + amino acid) appears in training
* aa_new — the position was seen in training, but only with a different amino acid substitution
* pos_novel — the mutation position was never observed in training

TabPFN was trained on the remaining variants and evaluated on each class separately using R², nRMSE, Spearman ρ, and top-k recall, averaged over 5 seeds (mean ± std).

In [20]:
MUT_RE = re.compile(r'^([A-Za-z])(\d+)([A-Za-z])$')

def parse_mutant(s):
    if pd.isna(s) or str(s).strip().upper() in ('', 'WT'):
        return {}
    d = {}
    for tok in str(s).split('-'):
        m = MUT_RE.match(tok.strip())
        if not m:
            raise ValueError(f"Cannot parse token {tok!r} in {s!r}")
        d[int(m.group(2))] = m.group(3).upper()
    return d

def build_variant_table(df):
    pos_sets, sub_sets, orders = [], [], []
    for s in df['mutant'].values:
        d = parse_mutant(s)
        pos_sets.append(frozenset(d.keys()))
        sub_sets.append(frozenset(d.items()))
        orders.append(len(d))
    orders = np.asarray(orders)

    single_row = {}
    for i, s in enumerate(sub_sets):
        if orders[i] == 1:
            single_row.setdefault(next(iter(s)), i)

    pair_to_rows = {}
    for i, s in enumerate(sub_sets):
        if orders[i] >= 2:
            for pair in combinations(sorted(s), 2):
                pair_to_rows.setdefault(pair, []).append(i)

    return dict(pos_sets=pos_sets, sub_sets=sub_sets, orders=orders,
                single_row=single_row, pair_to_rows=pair_to_rows)


class TrainIndex:
    def __init__(self, vt, train_idx):
        self.observed_subs      = set()
        self.observed_positions = set()
        for i in train_idx:
            for sub in vt['sub_sets'][i]:
                self.observed_subs.add(sub)
                self.observed_positions.add(sub[0])


def metric_block(y_true, y_pred, k_frac=0.1):
    k = max(1, int(k_frac * len(y_true)))
    true_topk = set(np.argsort(y_true)[-k:])
    pred_topk = set(np.argsort(y_pred)[-k:])
    return {
        'R2':          _r2(y_true, y_pred),
        'nRMSE':       _nrmse(y_true, y_pred),
        'Spearman':    spearmanr(y_true, y_pred).correlation,
        'topk_recall': len(true_topk & pred_topk) / k,
    }


def exp_position_vs_substitution_coverage(
        df, proteins,
        vt=None,
        pos_holdout_frac=0.15,
        single_holdout_frac=0.30,
        n_seeds=5,
        k_frac=0.1):

    if vt is None:
        vt = build_variant_table(df)

    y             = df['specific activity'].values.astype(float)
    orders        = vt['orders']
    single_rows   = np.where(orders == 1)[0]
    all_positions = sorted({p for ps in vt['pos_sets'] for p in ps})

    CLASSES = ['exact_seen', 'aa_new', 'pos_novel']
    rng_master = np.random.default_rng(0)
    rows = []

    for seed in range(n_seeds):
        rng = np.random.default_rng(rng_master.integers(1 << 30))

        # ── position hold-out ─────────────────────────────────────
        held_pos = set(rng.choice(
            all_positions,
            int(pos_holdout_frac * len(all_positions)),
            replace=False,
        ))

        singles_held    = [i for i in single_rows
                           if next(iter(vt['pos_sets'][i])) in held_pos]
        singles_nonheld = [i for i in single_rows
                           if next(iter(vt['pos_sets'][i])) not in held_pos]

        n_hold   = int(single_holdout_frac * len(singles_nonheld))
        test_idx = np.array(
            singles_held + list(rng.choice(singles_nonheld, n_hold, replace=False))
        )
        test_set = set(test_idx.tolist())

        train_idx = np.array([
            i for i in range(len(df))
            if not (vt['pos_sets'][i] & held_pos) and i not in test_set
        ])

        # ── test variant classification ───────────────────────────
        idx = TrainIndex(vt, train_idx)
        klass = []
        for i in test_idx:
            (sub,) = tuple(vt['sub_sets'][i])
            pos    = next(iter(vt['pos_sets'][i]))
            if sub in idx.observed_subs:
                klass.append('exact_seen')
            elif pos in idx.observed_positions:
                klass.append('aa_new')
            else:
                klass.append('pos_novel')
        klass = np.array(klass)

        # ── prediction & metrics ──────────────────────────────────
        model = TabPFNRegressor(device=TABPFN_DEVICE,
                                ignore_pretraining_limits=True,
                                random_state=seed)
        model.fit(proteins[train_idx], y[train_idx])
        y_pred = model.predict(proteins[test_idx])
        yt     = y[test_idx]

        cnt = {c: int((klass == c).sum()) for c in CLASSES}
        print(f"seed={seed}  train={len(train_idx)}  test={len(test_idx)}  {cnt}")

        for c in CLASSES:
            m = klass == c
            if m.sum() < 10:
                continue
            b = metric_block(yt[m], y_pred[m], k_frac)
            rows.append({
                'seed': seed, 'class': c, 'n': int(m.sum()),
                'R²': b['R2'], 'nRMSE': b['nRMSE'],
                'Spearman': b['Spearman'], 'top-k recall': b['topk_recall'],
            })

    raw_df = pd.DataFrame(rows)
    return raw_df


In [21]:
raw_df = exp_position_vs_substitution_coverage(df, proteins)

metrics = ['R²', 'nRMSE', 'Spearman', 'top-k recall']

summary_rows = []
for cls, g in raw_df.groupby('class'):
    row = {'class': cls, 'n': int(g['n'].mean())}
    for m in metrics:
        row[m] = f"{g[m].mean():.3f} ± {g[m].std():.3f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('class')
summary_df = summary_df.loc[['exact_seen', 'aa_new', 'pos_novel']]
display(summary_df)

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


seed=0  train=5241  test=3061  {'exact_seen': 28, 'aa_new': 1907, 'pos_novel': 1126}


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=1  train=4974  test=3059  {'exact_seen': 32, 'aa_new': 1903, 'pos_novel': 1124}
seed=2  train=5001  test=3045  {'exact_seen': 27, 'aa_new': 1914, 'pos_novel': 1104}
seed=3  train=5368  test=3046  {'exact_seen': 28, 'aa_new': 1913, 'pos_novel': 1105}


analytics-python queue is full


seed=4  train=5216  test=3052  {'exact_seen': 34, 'aa_new': 1904, 'pos_novel': 1114}


,n,R²,nRMSE,Spearman,top-k recall
class,,,,,
exact_seen,29,0.240 ± 0.184,0.867 ± 0.104,0.385 ± 0.179,0.267 ± 0.253
aa_new,1908,0.283 ± 0.039,0.846 ± 0.023,0.597 ± 0.013,0.291 ± 0.045
pos_novel,1114,0.162 ± 0.029,0.915 ± 0.016,0.423 ± 0.035,0.171 ± 0.020


## Exp 8. Exact substitution vs. Positional coverage
We investigate whether knowing the exact amino acid substitution (position + amino acid) provides predictive benefit beyond positional coverage alone. Using an extrapolation split (train on order ≤ 2, test on order > 2), we construct two size-matched training conditions: 

* (A) exact-match — variants sharing at least one mutation token (same position and amino acid) with the test set
* (B) position-only — variants covering the same positions but with a different amino acid substitution

To ensure a fair comparison, Train B is assembled by matching each Train A sample to a position-only sample with the maximally similar position set (exact position-set match first, then Jaccard similarity). Performance is then compared within bins of equal positional distance to the test set, isolating the contribution of substitution identity from positional proximity.

In [22]:
TRAIN_MAX_ORDER = 2
MIN_BIN_N       = 10
POS_RE          = re.compile(r'\d+')

tok_list = [frozenset(str(m).split(SEP)) for m in df['mutant']]
pos_list = [frozenset(int(POS_RE.search(t).group()) for t in toks if POS_RE.search(t)) for toks in tok_list]

y     = df[TARGET_COL].to_numpy(float)
order = df['num_mutations'].to_numpy()

test_idx   = np.where(order >  TRAIN_MAX_ORDER)[0]
train_pool = np.where(order <= TRAIN_MAX_ORDER)[0]

test_mut = set().union(*[tok_list[i] for i in test_idx])
test_pos = set().union(*[pos_list[i] for i in test_idx])

exact_idx    = [i for i in train_pool if tok_list[i] & test_mut]
pos_only_idx = [i for i in train_pool
                if not (tok_list[i] & test_mut) and (pos_list[i] & test_pos)]

print(f"exact={len(exact_idx)}  pos_only={len(pos_only_idx)}  test={len(test_idx)}")

def _multihot(sets, idxs, vocab):
    r, c = [], []
    for k, i in enumerate(idxs):
        for t in sets[i]:
            if t in vocab:
                r.append(k); c.append(vocab[t])
    return sparse.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(idxs), len(vocab)))

def min_sym_dist(sets, tr, te):
    vocab = {}
    for i in list(tr) + list(te):
        for t in sets[i]: vocab.setdefault(t, len(vocab))
    Xtr = _multihot(sets, tr, vocab)
    Xte = _multihot(sets, te, vocab)
    a = np.asarray(Xte.sum(1)).ravel()
    b = np.asarray(Xtr.sum(1)).ravel()
    out = np.empty(len(te), int)
    for s in range(0, len(te), 256):
        e = min(s + 256, len(te))
        D = a[s:e, None] + b[None, :] - 2 * (Xte[s:e] @ Xtr.T).toarray()
        out[s:e] = D.min(1)
    return out

def match_by_position(trainA, pos_only_idx, pos_list, rng):
    available = list(pos_only_idx)
    matched = []
    for i in trainA:
        pos_i = pos_list[i]
        same = [j for j in available if pos_list[j] == pos_i]
        chosen = rng.choice(same) if same else max(
            available,
            key=lambda j: len(pos_list[j] & pos_i) / max(len(pos_list[j] | pos_i), 1)
        )
        matched.append(int(chosen))
        available.remove(chosen)
    return matched

yt   = y[test_idx]
rows = []

for seed in SEEDS:
    rng = np.random.default_rng(seed)
    n   = min(len(exact_idx), len(pos_only_idx))

    trainA = list(rng.choice(exact_idx, n, replace=False))
    trainB = match_by_position(trainA, pos_only_idx, pos_list, rng)

    mdl_A = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=seed)
    mdl_A.fit(proteins[trainA], y[trainA])
    yA = np.asarray(mdl_A.predict(proteins[test_idx]))

    mdl_B = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=seed)
    mdl_B.fit(proteins[trainB], y[trainB])
    yB = np.asarray(mdl_B.predict(proteins[test_idx]))

    d_po_A = np.minimum(min_sym_dist(pos_list, trainA, test_idx), 5)
    d_po_B = np.minimum(min_sym_dist(pos_list, trainB, test_idx), 5)
    d_ex_A = np.minimum(min_sym_dist(tok_list, trainA, test_idx), 5)
    d_ex_B = np.minimum(min_sym_dist(tok_list, trainB, test_idx), 5)

    matched = (d_po_A == d_po_B)
    print(f"seed={seed}  n={n}  matched={matched.sum()}/{len(test_idx)}")

    for d in sorted(set(d_po_A[matched].tolist())):
        sel = matched & (d_po_A == d)
        if sel.sum() < MIN_BIN_N:
            continue
        rows.append({
            'seed': seed, 'dist_pos': d, 'n': int(sel.sum()),
            'A_R2':    _r2(yt[sel], yA[sel]),    'B_R2':    _r2(yt[sel], yB[sel]),
            'A_Sp':    _sp(yt[sel], yA[sel]),     'B_Sp':    _sp(yt[sel], yB[sel]),
            'A_nRMSE': _nrmse(yt[sel], yA[sel]), 'B_nRMSE': _nrmse(yt[sel], yB[sel]),
            'A_d_exact': float(d_ex_A[sel].mean()),
            'B_d_exact': float(d_ex_B[sel].mean()),
        })

res_df = pd.DataFrame(rows)

def fmt(g, col):
    return f"{g[col].mean():+.3f} ± {g[col].std():.3f}"

summary_rows = []
for d, g in res_df.groupby('dist_pos'):
    summary_rows.append({
        'pos_dist':     f"{d}" if d < 5 else "5+",
        'n':            f"{g['n'].mean():.0f} ± {g['n'].std():.0f}",
        'A (exact) R²':      fmt(g, 'A_R2'),
        'B (pos-only) R²':   fmt(g, 'B_R2'),
        'A Spearman':        fmt(g, 'A_Sp'),
        'B Spearman':        fmt(g, 'B_Sp'),
        'A nRMSE':           fmt(g, 'A_nRMSE'),
        'B nRMSE':           fmt(g, 'B_nRMSE'),
        'A d_exact (avg)':   round(g['A_d_exact'].mean(), 2),
        'B d_exact (avg)':   round(g['B_d_exact'].mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows).set_index('pos_dist')
display(summary_df)

analytics-python queue is full


exact=204  pos_only=1016  test=1086


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


seed=4192  n=204  matched=543/1086


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


...

error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=7060  n=204  matched=543/1086


analytics-python queue is full


seed=3597  n=204  matched=543/1086


,n,A (exact) R²,B (pos-only) R²,A Spearman,B Spearman,A nRMSE,B nRMSE,A d_exact (avg),B d_exact (avg)
pos_dist,,,,,,,,,
2,29 ± 0,+0.360 ± 0.013,+0.058 ± 0.048,+0.588 ± 0.009,+0.184 ± 0.141,+0.800 ± 0.008,+0.970 ± 0.025,2.0,4.07
3,23 ± 0,-0.217 ± 0.032,-0.174 ± 0.080,+0.293 ± 0.059,+0.187 ± 0.132,+1.103 ± 0.014,+1.083 ± 0.037,3.0,5.00
4,38 ± 0,-0.217 ± 0.030,-0.169 ± 0.076,+0.419 ± 0.016,+0.473 ± 0.086,+1.103 ± 0.013,+1.081 ± 0.035,4.0,5.00
5+,453 ± 0,-0.157 ± 0.048,-0.109 ± 0.044,+0.439 ± 0.038,+0.147 ± 0.075,+1.075 ± 0.022,+1.053 ± 0.021,5.0,5.00


## Exp 9. Position-holdout
We examine how prediction accuracy degrades as the number of completely unobserved mutation positions in a test variant increases. Two split strategies are evaluated under the same analysis framework:

* Position-holdout split — For each seed, 20% of all mutation positions are randomly held out. Variants containing at least one mutation at a held-out position form the test set; all remaining variants constitute the training set. By construction, every test variant contains at least one novel position.

* Extrapolation split — Training is restricted to low-order variants (order ≤ 2) and all higher-order variants (order > 2) form the test set.

In both settings, test variants are stratified by the count of positions that appear in the test variant but in no training variant (novel positions, ranging from 1 to 5+). A TabPFN model is evaluated on each stratum using R² and Spearman ρ, averaged over 5 seeds (mean ± std).

In [23]:
HOLDOUT_FRAC  = 0.20
MIN_N         = 15

POS_RE   = re.compile(r'\d+')
tok_list = [frozenset(str(m).split(SEP)) for m in df[MUT_COL]]
y        = df[TARGET_COL].to_numpy(float)

def pos_of(tok):
    m = POS_RE.search(str(tok))
    return int(m.group()) if m else None

all_pos = sorted({pos_of(t) for ts in tok_list for t in ts if pos_of(t) is not None})
print(f"unique positions = {len(all_pos)}")

rows = []

for seed in SEEDS:
    rng  = np.random.default_rng(seed)
    held = set(rng.choice(all_pos, int(HOLDOUT_FRAC * len(all_pos)), replace=False))

    nov_count = np.array([
        sum(1 for t in tok_list[i] if pos_of(t) in held)
        for i in range(len(df))
    ])
    train_idx = np.where(nov_count == 0)[0]
    test_idx  = np.where(nov_count >= 1)[0]
    nov       = nov_count[test_idx]

    print(f"seed={seed}  held={len(held)} pos  train={len(train_idx)}  test={len(test_idx)}")

    model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=seed)
    model.fit(proteins[train_idx], y[train_idx])
    pred = np.asarray(model.predict(proteins[test_idx]))
    yt   = y[test_idx]

    for k in range(1, 6):
        sel = (nov == k) if k < 5 else (nov >= 5)
        if sel.sum() < MIN_N:
            continue
        rows.append({
            'seed':    seed,
            'k':       k,                         # for sorting
            'n_novel': str(k) if k < 5 else '5+', # display label
            'n':       int(sel.sum()),
            'R2':      _r2(yt[sel], pred[sel]),
            'Spearman': _sp(yt[sel], pred[sel]),
        })

res_df = pd.DataFrame(rows)

def fmt(g, col):
    return f"{g[col].mean():+.3f} ± {g[col].std():.3f}"

summary = pd.DataFrame([
    {
        'n_novel':  g['n_novel'].iloc[0],
        'n':        f"{g['n'].mean():.0f} ± {g['n'].std():.0f}",
        'R²':       fmt(g, 'R2'),
        'Spearman': fmt(g, 'Spearman'),
    }
    for _, g in res_df.groupby('k')
]).set_index('n_novel')

summary.index.name = '# novel positions'
display(summary)

analytics-python queue is full


unique positions = 425
seed=4192  held=85 pos  train=6464  test=2301


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


seed=5638  held=85 pos  train=6322  test=2443


analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


...

analytics-python queue is full
analytics-python queue is full
analytics-python queue is full


seed=7060  held=85 pos  train=6476  test=2289


analytics-python queue is full
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


seed=3597  held=85 pos  train=6677  test=2088


,n,R²,Spearman
# novel positions,,,
1,1966 ± 79,+0.289 ± 0.146,+0.511 ± 0.034
2,205 ± 54,-0.125 ± 0.341,+0.335 ± 0.152
3,111 ± 36,-0.445 ± 0.125,+0.119 ± 0.114
4,52 ± 18,-1.215 ± 1.172,+0.107 ± 0.080


## Exp 10. Effect of pair-coverage within the same mutational distance
We examine whether mutation-pair coverage in the training set explains prediction accuracy for multi-mutation test variants, after controlling for mutational distance. Within each distance bin, multi-mutation test variants (order ≥ 2) are stratified by pair coverage — the fraction of their constituent mutation-token pairs that co-occur in at least one training variant. Full coverage (pc = 1.0) means every pairwise combination of the test variant's mutations was observed in training; partial coverage (pc < 1.0) means at least one pair is novel. R² and Spearman ρ are compared between the two strata, and the Spearman correlation between pair coverage and absolute prediction error quantifies the continuous relationship between pairwise training information and predictability.

In [24]:
TRAIN_MAX_ORDER = 2
TEST_FRAC       = 0.2
PAIR_FULL_THR   = 1.0
MIN_N           = 10

tok_list = [frozenset(str(m).split(SEP)) for m in df[MUT_COL]]
y        = df[TARGET_COL].to_numpy(float)
order    = df[ORDER_COL].to_numpy(int)
N        = len(df)

vocab = {}
for ts in tok_list:
    for t in ts: vocab.setdefault(t, len(vocab))

def _multihot(idxs):
    r, c = [], []
    for k, i in enumerate(idxs):
        for t in tok_list[i]: r.append(k); c.append(vocab[t])
    return sparse.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(idxs), len(vocab)))

def _pair_cov(i, train_pairs):
    ps = list(combinations(sorted(tok_list[i]), 2))
    return np.nan if not ps else np.mean([frozenset(p) in train_pairs for p in ps])

def run_a3(split_mode):
    rows = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed)

        if split_mode == 'extrapolate':
            test_idx  = np.where(order >  TRAIN_MAX_ORDER)[0]
            train_idx = np.where(order <= TRAIN_MAX_ORDER)[0]
        else:
            _test = []
            for o in np.unique(order):
                io = np.where(order == o)[0]
                nt = int(round(TEST_FRAC * len(io)))
                if len(io) > 1 and nt > 0:
                    _test += list(rng.choice(io, nt, replace=False))
            test_idx  = np.array(sorted(_test))
            mask      = np.ones(N, bool); mask[test_idx] = False
            train_idx = np.where(mask)[0]

        print(f"[{split_mode}] seed={seed}  train={len(train_idx)}  test={len(test_idx)}")

        model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=int(seed))
        model.fit(proteins[train_idx], y[train_idx])
        pred   = np.asarray(model.predict(proteins[test_idx]))
        yt     = y[test_idx]
        ot     = order[test_idx]
        abserr = np.abs(yt - pred)

        Xtr, Xte = _multihot(train_idx), _multihot(test_idx)
        a_ = np.asarray(Xte.sum(1)).ravel()
        b_ = np.asarray(Xtr.sum(1)).ravel()
        inter    = (Xte @ Xtr.T).toarray()
        min_dist = (a_[:, None] + b_[None, :] - 2 * inter).min(axis=1).astype(int)
        del inter
        dbin = np.minimum(min_dist, 5)

        train_pairs = {frozenset(p) for i in train_idx
                       for p in combinations(sorted(tok_list[i]), 2)}
        pc = np.array([_pair_cov(i, train_pairs) for i in test_idx])

        for d in sorted(set(dbin)):
            seld = (dbin == d) & (ot >= 2) & ~np.isnan(pc)
            if seld.sum() < MIN_N:
                continue
            full = seld & (pc >= PAIR_FULL_THR)
            part = seld & (pc <  PAIR_FULL_THR)
            rho  = spearmanr(pc[seld], abserr[seld]).correlation \
                   if seld.sum() >= 30 and np.std(pc[seld]) > 0 else np.nan

            rows.append({
                'seed':       seed,
                'dist':       d,
                'n_full':     int(full.sum()),
                'n_part':     int(part.sum()),
                'R2_full':    _r2(yt[full], pred[full]) if full.sum() >= MIN_N else np.nan,
                'Sp_full':    _sp(yt[full], pred[full]) if full.sum() >= MIN_N else np.nan,
                'R2_part':    _r2(yt[part], pred[part]) if part.sum() >= MIN_N else np.nan,
                'Sp_part':    _sp(yt[part], pred[part]) if part.sum() >= MIN_N else np.nan,
                'rho_pc_err': rho,
            })
    return pd.DataFrame(rows)

def make_summary(res_df):
    def fmt(g, col):
        m, s = g[col].mean(), g[col].std()
        return f"{m:+.3f} ± {s:.3f}" if not np.isnan(m) else "—"
    def fmt_n(g, col):
        return f"{g[col].mean():.0f} ± {g[col].std():.0f}"

    return pd.DataFrame([
        {
            'mut_dist':             f"{d}" if d < 5 else "5+",
            'n (full)':             fmt_n(g, 'n_full'),
            'R² (full)':            fmt(g, 'R2_full'),
            'Spearman (full)':      fmt(g, 'Sp_full'),
            'n (partial)':          fmt_n(g, 'n_part'),
            'R² (partial)':         fmt(g, 'R2_part'),
            'Spearman (partial)':   fmt(g, 'Sp_part'),
            'ρ(pair_cov, |error|)': fmt(g, 'rho_pc_err'),
        }
        for d, g in res_df.groupby('dist')
    ]).set_index('mut_dist')


for split_mode in ['random', 'extrapolate']:
    res_df = run_a3(split_mode)
    print(f"\n{'='*60}")
    print(f"  Split: {split_mode.upper()}")
    print(f"{'='*60}")
    display(make_summary(res_df))

[random] seed=4192  train=7012  test=1753
[random] seed=5638  train=7012  test=1753


analytics-python queue is full


[random] seed=3285  train=7012  test=1753


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


[random] seed=7060  train=7012  test=1753
[random] seed=3597  train=7012  test=1753

  Split: RANDOM


...

analytics-python queue is full


[extrapolate] seed=3597  train=7679  test=1086


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)



  Split: EXTRAPOLATE


,n (full),R² (full),Spearman (full),n (partial),R² (partial),Spearman (partial),"ρ(pair_cov, |error|)"
mut_dist,,,,,,,
1,18 ± 0,+0.244 ± 0.033,+0.768 ± 0.040,107 ± 0,+0.376 ± 0.008,+0.663 ± 0.009,-0.153 ± 0.008
2,2 ± 0,—,—,190 ± 0,+0.260 ± 0.009,+0.569 ± 0.016,-0.134 ± 0.014
3,0 ± 0,—,—,162 ± 0,+0.087 ± 0.014,+0.416 ± 0.024,-0.215 ± 0.011
4,0 ± 0,—,—,123 ± 0,-0.026 ± 0.021,+0.379 ± 0.040,+0.010 ± 0.032
5+,0 ± 0,—,—,484 ± 0,-0.245 ± 0.034,-0.056 ± 0.103,+0.134 ± 0.015


## Exp 11. Prediction accuracy and mutational position coverage
We evaluate prediction accuracy as a function of positional distance to the nearest training variant — defined as the symmetric difference in mutated position sets, ignoring the specific amino acid at each position. A positional distance of 0 indicates that some training variant covers the exact same set of mutated positions (though possibly with different amino acid substitutions); distance 1 indicates that the nearest training variant differs by one position, and so on. R², Spearman ρ, and nRMSE are reported per distance bin under both a random 80/20 split and an order-based extrapolation split (train ≤ 2, test > 2), averaged over 5 seeds (mean ± std).

In [25]:
TRAIN_MAX_ORDER = 2
TEST_FRAC       = 0.2
MIN_N           = 15
MAX_DIST        = 3     # 3 means "3+"
POS_RE   = re.compile(r'\d+')

pos_list = [frozenset(int(POS_RE.findall(t)[0]) for t in str(m).split(SEP))
            for m in df[MUT_COL]]
y     = df[TARGET_COL].to_numpy(float)
order = df[ORDER_COL].to_numpy(int)
N     = len(df)

def _pos_dist(train_idx, test_idx):
    v = {}
    for i in list(train_idx) + list(test_idx):
        for p in pos_list[i]: v.setdefault(p, len(v))

    def mh(idxs):
        r, c = [], []
        for k, i in enumerate(idxs):
            for p in pos_list[i]: r.append(k); c.append(v[p])
        return sparse.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(idxs), len(v)))

    Xtr, Xte = mh(train_idx), mh(test_idx)
    a = np.asarray(Xte.sum(1)).ravel()
    b = np.asarray(Xtr.sum(1)).ravel()
    out = np.empty(len(test_idx), int)
    for s in range(0, len(test_idx), 256):
        e = min(s + 256, len(test_idx))
        D = a[s:e, None] + b[None, :] - 2 * (Xte[s:e] @ Xtr.T).toarray()
        out[s:e] = D.min(1)
    return out

def run_b(split_mode):
    rows = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed)

        if split_mode == 'extrapolate':
            test_idx  = np.where(order >  TRAIN_MAX_ORDER)[0]
            train_idx = np.where(order <= TRAIN_MAX_ORDER)[0]
        else:
            perm      = rng.permutation(N)
            nt        = int(TEST_FRAC * N)
            test_idx  = perm[:nt]
            train_idx = perm[nt:]

        print(f"[{split_mode}] seed={seed}  train={len(train_idx)}  test={len(test_idx)}")

        model = TabPFNRegressor(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=seed)
        model.fit(proteins[train_idx], y[train_idx])
        pred = np.asarray(model.predict(proteins[test_idx]))
        yt   = y[test_idx]

        d_po = _pos_dist(train_idx, test_idx)
        dbin = np.minimum(d_po, MAX_DIST)

        for d in range(MAX_DIST + 1):
            sel = (dbin == d)
            if sel.sum() < MIN_N:
                continue
            rows.append({
                'seed':  seed,
                'dist':  d,
                'label': str(d) if d < MAX_DIST else f'{MAX_DIST}+',
                'n':     int(sel.sum()),
                'R2':    _r2(yt[sel], pred[sel]),
                'Sp':    _sp(yt[sel], pred[sel]),
                'nRMSE': _nrmse(yt[sel], pred[sel]),
            })
    return pd.DataFrame(rows)


def make_summary(res_df):
    def fmt(g, col):
        m, s = g[col].mean(), g[col].std()
        return f"{m:+.3f} ± {s:.3f}" if not np.isnan(m) else "—"

    return pd.DataFrame([
        {
            'pos_dist': g['label'].iloc[0],
            'n':        f"{g['n'].mean():.0f} ± {g['n'].std():.0f}",
            'R²':       fmt(g, 'R2'),
            'Spearman': fmt(g, 'Sp'),
            'nRMSE':    fmt(g, 'nRMSE'),
        }
        for _, g in res_df.groupby('dist')
    ]).set_index('pos_dist')


for split_mode in ['random', 'extrapolate']:
    res_df = run_b(split_mode)
    print(f"\n{'='*55}\n  Split: {split_mode.upper()}\n{'='*55}")
    display(make_summary(res_df))


[random] seed=4192  train=7012  test=1753
[random] seed=5638  train=7012  test=1753
[random] seed=3285  train=7012  test=1753


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)


[random] seed=7060  train=7012  test=1753
[random] seed=3597  train=7012  test=1753

  Split: RANDOM


,n,R²,Spearman,nRMSE
pos_dist,,,,
0,1571 ± 7,+0.389 ± 0.140,+0.589 ± 0.023,+0.778 ± 0.089
1,145 ± 10,+0.698 ± 0.080,+0.811 ± 0.057,+0.546 ± 0.069
2,29 ± 6,+0.622 ± 0.143,+0.710 ± 0.136,+0.603 ± 0.136


[extrapolate] seed=4192  train=7679  test=1086


...

[extrapolate] seed=3285  train=7679  test=1086
[extrapolate] seed=7060  train=7679  test=1086
[extrapolate] seed=3597  train=7679  test=1086


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)



  Split: EXTRAPOLATE


,n,R²,Spearman,nRMSE
pos_dist,,,,
1,134 ± 0,+0.375 ± 0.011,+0.688 ± 0.011,+0.791 ± 0.007
2,195 ± 0,+0.273 ± 0.007,+0.596 ± 0.013,+0.853 ± 0.004
3+,757 ± 0,-0.160 ± 0.030,+0.135 ± 0.072,+1.077 ± 0.014


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Read timed out. (read timeout=15)
